In [1]:
%matplotlib inline

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from qiskit import QuantumCircuit, transpile
try:
    from iqm.qiskit import IQMProvider
except ImportError:
    from iqm.qiskit_iqm import IQMProvider

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, roc_curve,
    balanced_accuracy_score, matthews_corrcoef
)
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. KONFIGURACJA I PRZYCZYNOWE SKALOWANIE (0% DATA LEAKAGE)
# ==========================================
csv_path = "dataset_final.csv" if os.path.exists("dataset_final.csv") else "dane/dataset_final.csv"

# Topologia Gwiazdy: Kubit 1 to HUB, reszta to satelity
FEATURES_MAP = [
    'pbv_pko', 'wibor_3m',         # Kubit 0 (Satelita)
    'pbv_peo', 'rentownosc_10y',   # Kubit 1 (HUB)
    'pbv_san', 'eurpln',           # Kubit 2 (Satelita)
    'pbv_ing', 'wig20',            # Kubit 3 (Satelita)
    'pbv_mbk', 'zmiennosc'         # Kubit 4 (Satelita)
]

df = pd.read_csv(csv_path, index_col=0)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

X_raw = df[FEATURES_MAP].values
y_raw = df["target_y"].astype(int).values
dates_raw = df.index.values

LOOKBACK = 52
X_norm_list, valid_indices = [], []

# Rygorystyczne skalowanie w oparciu wyłącznie o przeszłość (okno 52 tygodni)
for t in range(LOOKBACK, len(X_raw)):
    past_window = X_raw[t - LOOKBACK : t]
    min_vals = past_window.min(axis=0)
    max_vals = past_window.max(axis=0)
    range_vals = np.where((max_vals - min_vals) == 0, 1e-8, max_vals - min_vals)
    
    current_scaled = ((X_raw[t] - min_vals) / range_vals) * (2 * np.pi)
    current_scaled = np.clip(current_scaled, 0.0, 2 * np.pi)
    
    X_norm_list.append(current_scaled)
    valid_indices.append(t)

X_quantum = np.array(X_norm_list)
y = y_raw[valid_indices]
dates = dates_raw[valid_indices]
X_raw_valid = X_raw[valid_indices]

print(f">>> Causal Scaling zakończony. Tygodni Walk-Forward: {len(X_quantum)}")

# ==========================================
# 2. MECHANIZM RZUTÓW Z FIZYCZNEGO QPU ODRA
# ==========================================
HUB_QUBIT = 1

def build_base_ansatz(x, num_qubits=5, hub_qubit=HUB_QUBIT):
    qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        qc.ry(x[2 * i], i)
        qc.rz(x[2 * i + 1], i)
    for i in range(num_qubits):
        if i != hub_qubit:
            qc.cx(hub_qubit, i)
    return qc

def build_measurement_circuits(x, num_qubits=5, hub_qubit=HUB_QUBIT):
    qc_x = build_base_ansatz(x, num_qubits, hub_qubit)
    for i in range(num_qubits): qc_x.h(i)
    qc_x.measure_all()
    
    qc_y = build_base_ansatz(x, num_qubits, hub_qubit)
    for i in range(num_qubits): 
        qc_y.sdg(i)
        qc_y.h(i)
    qc_y.measure_all()
    
    qc_z = build_base_ansatz(x, num_qubits, hub_qubit)
    qc_z.measure_all()
    return qc_x, qc_y, qc_z

def expectation_from_counts(counts, num_qubits):
    total_shots = sum(counts.values())
    expvals = np.zeros(num_qubits)
    for bitstring, count in counts.items():
        bits = [1 if b == '0' else -1 for b in reversed(bitstring.replace(" ", ""))]
        for q in range(num_qubits): expvals[q] += bits[q] * count
    return expvals / total_shots

def get_secure_qpu_projections(X_q, dates_arr, features_arr, filepath, num_qubits=5, reps=1, shots=4000, batch_size=50):
    if os.path.exists(filepath):
        print(f">>> Błyskawiczne ładowanie zwalidowanego cache QPU: {filepath}")
        cache = np.load(filepath, allow_pickle=True)
        if not np.array_equal(cache["dates"], dates_arr): 
            raise ValueError("Mismatch dat w cache. Należy usunąć plik i wygenerować rzuty ponownie.")
        return cache["phi"]
        
    print(">>> Brak zwalidowanego cache. Łączenie z QPU Odra (IQM Spark)...")
    env_paths = ["../iqm_token.env", "iqm_token.env", "token.env", "../token.env"]
    for path in env_paths:
        if os.path.exists(path):
            load_dotenv(path)
            break
            
    server_url = os.getenv("SERVER")
    provider = IQMProvider(server_url)
    backend = provider.get_backend()
    
    all_circuits = []
    for x in X_q: all_circuits.extend(build_measurement_circuits(x, num_qubits, HUB_QUBIT))
    
    transpiled = transpile(all_circuits, backend=backend, optimization_level=2)
    all_counts = []
    
    for i in range(0, len(transpiled), batch_size):
        batch = transpiled[i : i + batch_size]
        print(f"  Wysyłanie paczki {i//batch_size + 1} / {(len(transpiled)-1)//batch_size + 1}...")
        job = backend.run(batch, shots=shots)
        for j in range(len(batch)): all_counts.append(job.result().get_counts(j))
            
    Phi_odra = np.zeros((len(X_q), num_qubits * 3))
    for i in range(len(X_q)):
        for q in range(num_qubits):
            Phi_odra[i, 3 * q + 0] = expectation_from_counts(all_counts[3 * i], num_qubits)[q]
            Phi_odra[i, 3 * q + 1] = expectation_from_counts(all_counts[3 * i + 1], num_qubits)[q]
            Phi_odra[i, 3 * q + 2] = expectation_from_counts(all_counts[3 * i + 2], num_qubits)[q]
            
    np.savez(filepath, phi=Phi_odra, dates=dates_arr, features=np.array(features_arr), reps=reps, shots=shots, num_qubits=num_qubits, backend=backend.name)
    return Phi_odra

# Pobranie wyliczonych już rzutów z Twojego dysku
cache_filename = "phi_odra_wigbanki_5q_star_reps1_shots4000_v1.npz"
Phi_real = get_secure_qpu_projections(X_quantum, dates, FEATURES_MAP, cache_filename, reps=1)

# ==========================================
# 3. KROCZĄCY TRENING MODELU QPU (Z PLATT SCALINGIEM)
# ==========================================
# KLUCZOWY ELEMENT: probability=True włącza Platt Scaling, co generuje płynne prawdopodobieństwa
def make_q_svm_pipeline():
    return Pipeline([("scaler", StandardScaler()), ("svm", SVC(kernel="rbf", probability=True, random_state=42))])

param_grid = {"svm__C": [0.1, 1.0, 5.0, 10.0], "svm__gamma": ["scale", "auto"]}
inner_cv = TimeSeriesSplit(n_splits=3)
TRAIN_WINDOW = 52
n_samples = len(y)

results = {"y_true": [], "dates": [], "prob_qpu": []}

print("\n>>> Start ewaluacji Walk-Forward dla Quantum SVM (Standalone)...")

for t in range(TRAIN_WINDOW, n_samples):
    P_tr, P_te = Phi_real[t - TRAIN_WINDOW : t], Phi_real[t : t + 1]
    y_tr = y[t - TRAIN_WINDOW : t]
    
    results["y_true"].append(y[t])
    results["dates"].append(dates[t])
    
    # GridSearch uczy się na małym oknie, wyciągając najlepsze hiperparametry
    search_qpu = GridSearchCV(make_q_svm_pipeline(), param_grid, cv=inner_cv, scoring="roc_auc", n_jobs=1)
    search_qpu.fit(P_tr, y_tr)
    
    # Pobieramy precyzyjnie wygładzone prawdopodobieństwo Platt'a
    results["prob_qpu"].append(search_qpu.predict_proba(P_te)[0, 1])

for k in results: results[k] = np.array(results[k])

y_true = results["y_true"]
prob_qpu = results["prob_qpu"]
pred_qpu = (prob_qpu > 0.5).astype(int) # Dynamiczne cięcie na 0.5 zamiast 0

# ==========================================
# 4. METRYKI I WIZUALIZACJA
# ==========================================
def compute_metrics(y_t, pred, probs):
    return {
        "Accuracy": accuracy_score(y_t, pred),
        "Balanced Acc": balanced_accuracy_score(y_t, pred),
        "F1": f1_score(y_t, pred, zero_division=0),
        "AUC": roc_auc_score(y_t, probs),
        "MCC": matthews_corrcoef(y_t, pred)
    }

metrics = compute_metrics(y_true, pred_qpu, prob_qpu)
df_metrics = pd.DataFrame({"Ultimate QPU Odra SVM (Standalone)": metrics}).T

print("\n--- FINALNE WYNIKI PUBLIKACYJNE ---")
display(df_metrics.style.format("{:.4f}"))

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
fpr, tpr, _ = roc_curve(y_true, prob_qpu)

# Wykres ROC
ax[0].plot(fpr, tpr, color='#8e44ad', lw=2.5, label=f'QPU SVM (AUC = {metrics["AUC"]:.4f})')
ax[0].plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Random Guess')
ax[0].set_title('Krzywa ROC: Kwantowa Zdolność Rozdzielcza', fontweight='bold', fontsize=12)
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')
ax[0].legend(loc='lower right')
ax[0].grid(True, linestyle='--', alpha=0.6)

# Wykres Equity / Skumulowany Bilans Trafień
net_hits = np.cumsum(np.where(pred_qpu == y_true, 1, -1))
ax[1].plot(results["dates"], net_hits, color='#8e44ad', lw=2.5, label='QPU SVM (Topologia Gwiazdy)')
ax[1].axhline(0, color='black', lw=1.5)
ax[1].set_title('Skumulowany Bilans Trafień (Walk-Forward OOS)', fontweight='bold', fontsize=12)
ax[1].set_xlabel('Data')
ax[1].set_ylabel('Net Hits (Trafienia - Błędy)')
ax[1].legend(loc='upper left')
ax[1].grid(True, linestyle='--', alpha=0.6)

# Gradient wypełniający zyski pod krzywą kapitału dla estetyki
ax[1].fill_between(results["dates"], 0, net_hits, where=(net_hits > 0), color='#8e44ad', alpha=0.1)
ax[1].fill_between(results["dates"], 0, net_hits, where=(net_hits <= 0), color='red', alpha=0.1)

plt.tight_layout()
plt.show()

: 